In [ ]:
# Cell 1: Imports and load the project's data/graph
import pandas as pd
import networkx as nx
import igraph as ig
from pathlib import Path

# Adjust these paths to match the small or full dataset you want to use
dataset_path = Path("dataset/small-dataset")   # or "full-dataset"

# Load nodes and edges — adapt based on the exact CSV/Excel names in the dataset folder
nodes_df = pd.read_csv(dataset_path / "nodes.csv")      # example filename — check actual files
edges_df = pd.read_csv(dataset_path / "edges.csv")      # example — check actual files

# Build NetworkX graph (the repo's style)
G_nx = nx.Graph()
for _, row in nodes_df.iterrows():
    G_nx.add_node(row['node_id'], **row.to_dict())

for _, row in edges_df.iterrows():
    G_nx.add_edge(row['from_node'], row['to_node'], 
                  cost=row.get('cost', row.get('length', 1)),
                  delay=row.get('delay', row.get('length', 1)/1000),
                  capacity=row.get('capacity', 100))

print(f"Loaded graph: {G_nx.number_of_nodes()} nodes, {G_nx.number_of_edges()} edges")

# Convert to igraph for fast disjoint-path computation
G_ig = ig.Graph.from_networkx(G_nx)
G_ig.vs["name"] = [str(v) for v in G_nx.nodes()]
print("Graph ready for redundant pathway optimisation ✓")

In [ ]:
# Cell 2: Your core algorithm — Redundant Pathways Optimiser (Stage 1)
def find_l_edge_disjoint_paths(G_ig, source_name, target_name, L=4):
    source = G_ig.vs.find(name=str(source_name)).index
    target = G_ig.vs.find(name=str(target_name)).index
    paths = []
    Gc = G_ig.copy()
    for _ in range(L):
        try:
            path = Gc.get_shortest_paths(source, target, weights="delay", output="vpath")[0]
            if not path:
                break
            paths.append([Gc.vs[v]["name"] for v in path])
            edge_ids = Gc.get_eids(path=path)
            Gc.delete_edges(edge_ids)
        except:
            break
    return paths

# Pick any two nodes from the project's map (use node IDs shown when you hover or print G_nx.nodes())
paths = find_l_edge_disjoint_paths(G_ig, source_name=123456, target_name=789012, L=4)   # ← change these IDs
print(f"Found {len(paths)} edge-disjoint paths:")
for i, p in enumerate(paths):
    print(f"Path {i+1}: {' → '.join(map(str, p))}")